In [1]:
RERUN = False          # DNA_RERUN=1 in the environment turns the GPU rerun cells on
from _common import *
from _dna_helpers import *

RERUN = rerun_flag(RERUN)
P = paper()

# 02 · DNA base model, dMFM training, split and C0 regressors (Tables 7–11)

Paper App. E.1 (Tables 7–9), App. F.1 (Table 10), Fig 3 / Table 11.
Hyperparameters are read from the `args.json` files stored next to each checkpoint (copied to
`results/dna/training/`); step counts and parameter counts were read from the checkpoints themselves
(`training/checkpoint_summary.json`, extracted with `torch.load` from the backup copies).

| Paper item | Status |
|---|---|
| Table 7 | architecture and parameter counts reproduced exactly; **loss weights 0.35 / 10 are wrong (all runs used 1.0 / 1.0)**; "GLASS NFE 8" is not a training argument; "10 hours" training time not supported (runs took ≈ 0.7 h) |
| Table 8 | caption only, no entries in the paper |
| Table 9 | reproduced, with two clarifications: base "200,000 steps" is the run length but the checkpoint used everywhere is `best.pt` from step 45–55k; the L = 400 dMFM's last 750 steps were a diagonal-only fine-tune at batch 8 |
| Table 10 | reproduced exactly |
| Table 11 | reproduced exactly (from the 8,294 stored test predictions) |
| Fig 6, Table 16 | **moved** to `notebooks/09_dna_regenerated.ipynb` (dna-regen agent; regenerated) |

In [2]:
T = DNA / "training"
rd = lambda name: json.loads((T / name).read_text())
ck = rd("checkpoint_summary.json")
split = rd("parent_split_seed0.json")
base = {L: rd(f"base_L{L}_args.json") for L in (50, 100, 200, 400)}
dmfm = {L: rd(f"dmfm_L{L}_args.json") for L in (50, 100, 200, 400)}
dmfm400_parent = rd("dmfm_L400_parent_esd_run_args.json")

## Table 7 — dMFM hyperparameters (L = 50)

The dMFM is built on the teacher's architecture config (`base_L50_args.json → model_cfg`); the dMFM run's own
args give the gap range, loss type and loss weights. Parameter counts are summed over the `state_dict` of the
selected checkpoint `yeast_parent_a_L50_dmfm_diagonal_h192_b4/epoch=95-step=49000.ckpt`:
DiT backbone = `model.model.*`; embeddings = `x_embed` + `x_cond_embed`; x_cond gate = `x_cond_adaLN`;
time embedders = the five `{s,t,t_cond,s_second,t_second}_embedder` MLPs.

The ESD consistency loss is multiplied by `mfm_lsd_loss_weight` (`utils/mfm_diag_distill.py`, line ~404), the
diagonal loss by `mfm_diag_loss_weight`. Both are 1.0 in every surviving dMFM run (all four lengths, the L = 400
fine-tune, and the 4-step students), not 0.35 / 10 as printed.

In [3]:
cfg = base[50]["model_cfg"]
a = dmfm[50]
g = ck["dmfm"]["50"]["param_groups"]
p7 = P["table7_dmfm_hparams"]
emb = g["model.x_embed.embedding"] + g["model.x_cond_embed.embedding"]
time_emb = sum(v for k, v in g.items() if k.endswith("embedder.mlp") or k.endswith("embedder_second.mlp"))
backbone = sum(v for k, v in g.items() if k.startswith("model.model."))
rows = [
    dict(item="sequence length L", paper=p7["seq_len"], ours=cfg["seq_len"], atol=0),
    dict(item="alphabet size K", paper=p7["alphabet_size"], ours=cfg["alphabet_size"], atol=0),
    dict(item="gap range min", paper=p7["gap_range"][0], ours=a["mfm_consistency_min_gap"], dec=2),
    dict(item="gap range max", paper=p7["gap_range"][1], ours=a["mfm_consistency_max_gap"], dec=1),
    dict(item="hidden dim d", paper=p7["hidden_dim"], ours=a["hidden_dim"], atol=0),
    dict(item="transformer blocks", paper=p7["transformer_blocks"], ours=a["dit_blocks"], atol=0),
    dict(item="attention heads", paper=p7["attention_heads"], ours=a["transformer_heads"], atol=0),
    dict(item="head dim", paper=p7["head_dim"], ours=a["hidden_dim"] // a["transformer_heads"], atol=0),
    dict(item="MLP expansion ratio", paper=p7["mlp_ratio"], ours=float(a["transformer_ff_mult"]), dec=1),
    dict(item="conditioning dim", paper=p7["cond_dim"], ours=cfg["dit_cond_dim"], atol=0),
    dict(item="attention softcap", paper=p7["attention_softcap"], ours=cfg["softcap"], dec=1),
    dict(item="total parameters", paper=p7["params_total"], ours=ck["dmfm"]["50"]["n_params_total"], atol=0),
    dict(item="DiT backbone parameters", paper=p7["params_dit_backbone"], ours=backbone, atol=0,
         note=f"= base generator size {ck['base']['50']['best.pt']['n_params']:,}"),
    dict(item="embedding parameters", paper=p7["params_embeddings"], ours=emb, atol=0),
    dict(item="x_cond gate parameters", paper=p7["params_xcond_gate"], ours=g["model.x_cond_adaLN.1"], atol=0),
    dict(item="time embedder parameters", paper=p7["params_time_embedders"], ours=time_emb, atol=0),
    dict(item="diagonal loss weight", paper=p7["diagonal_weight"], ours=a["mfm_diag_loss_weight"], dec=2, note="KNOWN text error: every run used 1.0"),
    dict(item="consistency (ESD) loss weight", paper=p7["consistency_weight"], ours=a["mfm_lsd_loss_weight"], dec=1, note="KNOWN text error: every run used 1.0"),
    dict(item="GLASS NFE", paper=p7["glass_nfe"], ours=None, note="not a training argument; no surviving arg equals it unambiguously (Table 1 sampling uses mc=8 posterior samples)"),
]
t7 = show(cmp(rows))
print("consistency loss type:", a["mfm_consistency_type"], "(paper: ESD loss)")
t7

status counts: {'match': 16, 'MISMATCH': 2, 'n/a': 1}
consistency loss type: esd_teacher (paper: ESD loss)


,item,paper,ours,status,tol,tol_src,note
0,sequence length L,50.00,50,match,0,atol,
1,alphabet size K,4.00,4,match,0,atol,
2,gap range min,0.75,0.75,match,0.005,atol,
3,gap range max,1.00,1.0,match,0.05,atol,
4,hidden dim d,192.00,192,match,0,atol,
5,transformer blocks,4.00,4,match,0,atol,
6,attention heads,6.00,6,match,0,atol,
7,head dim,32.00,32,match,0,atol,
8,MLP expansion ratio,4.00,4.0,match,0.05,atol,
9,conditioning dim,192.00,192,match,0,atol,


In [4]:
# every surviving dMFM training run used weights 1.0 / 1.0
pd.DataFrame([{"run": name, "diag_w": d["mfm_diag_loss_weight"], "consistency_w": d["mfm_lsd_loss_weight"],
               "consistency": d["mfm_consistency"], "type": d["mfm_consistency_type"], "batch": d["batch_size"],
               "lr": d["lr"], "max_steps": d["max_steps"]}
              for name, d in [*((f"dmfm L{L}", dmfm[L]) for L in (50, 100, 200)), ("dmfm L400 parent ESD run", dmfm400_parent),
                              ("dmfm L400 selected (diag-only fine-tune)", dmfm[400]),
                              *((f"4-step student L{L}", rd(f"dmfm_4step_L{L}_args.json")) for L in (50, 100, 200, 400))]])

,run,diag_w,consistency_w,consistency,type,batch,lr,max_steps
0,dmfm L50,1.0,1.0,True,esd_teacher,64,0.00010,50000
1,dmfm L100,1.0,1.0,True,esd_teacher,64,0.00010,50000
2,dmfm L200,1.0,1.0,True,esd_teacher,32,0.00010,50000
3,dmfm L400 parent ESD run,1.0,1.0,True,esd_teacher,16,0.00010,50000
4,dmfm L400 selected (diag-only fine-tune),1.0,0.0,False,lsd,8,0.00010,46000
5,4-step student L50,1.0,1.0,True,esd_teacher,64,0.00002,150000
6,4-step student L100,1.0,1.0,True,esd_teacher,64,0.00002,150000
7,4-step student L200,1.0,1.0,True,esd_teacher,32,0.00002,150000
8,4-step student L400,1.0,1.0,True,esd_teacher,16,0.00002,150000


**Training time.** App. E.1 says "We trained the dMFM for 10 hours on one 80GB NVIDIA H100 GPU". The surviving runs
are much shorter: from `args.json` to the last checkpoint (file mtimes, preserved in the backup; the tfevents start
time agrees) each parent-disjoint dMFM run took ≈ 0.6–0.7 h for 50,000 steps, and each base generator ≈ 1 h for
200,000 steps. The 10 h figure may describe the older `split65k` dMFM (632k steps), which is not what the paper's
DNA results use. **New discrepancy** (text only).

In [5]:
wc = rd("wallclock_from_mtimes.json")
wct = pd.DataFrame(wc).T
display(wct)
show(cmp([dict(item="dMFM L50 training wall-clock (h)", paper=10, ours=wc["yeast_parent_a_L50_dmfm_diagonal_h192_b4"]["wall_hours_args_to_last_ckpt"], atol=0.5,
               note="NEW: surviving run took ~0.7 h (mtime-based)")]))

,args_json_mtime,tfevents_start,last_checkpoint,last_checkpoint_mtime,wall_hours_args_to_last_ckpt
yeast_parent_a_L50_dfm_dit_h192_b4,2026-07-25T19:39:45,None,last.pt,2026-07-25T20:38:39,0.982
yeast_parent_a_L100_dfm_dit_h192_b4,2026-07-25T19:39:45,None,last.pt,2026-07-25T20:37:41,0.965
yeast_parent_a_L200_dfm_dit_h192_b4,2026-07-25T19:39:44,None,last.pt,2026-07-25T20:36:46,0.95
yeast_parent_a_L400_dfm_dit_h192_b4,2026-07-25T19:39:44,None,last.pt,2026-07-25T20:52:30,1.213
yeast_parent_a_L50_dmfm_diagonal_h192_b4,2026-07-26T09:32:02,2026-07-26T09:32:50,last.ckpt,2026-07-26T10:13:37,0.693
yeast_parent_a_L100_dmfm_diagonal_h192_b4,2026-07-26T09:32:02,2026-07-26T09:32:50,last.ckpt,2026-07-26T10:12:36,0.676
yeast_parent_a_L200_dmfm_diagonal_h192_b4,2026-07-26T09:32:02,2026-07-26T09:32:50,last.ckpt,2026-07-26T10:09:39,0.627
yeast_parent_a_L400_dmfm_diagonal_h192_b4,2026-07-26T09:32:12,2026-07-26T09:33:05,last.ckpt,2026-07-26T10:10:30,0.639
yeast_parent_a_L400_dmfm_diagonly_finetune_20260726,2026-07-26T16:12:12,2026-07-26T16:11:45,last-v1.ckpt,2026-07-26T16:12:58,0.013


status counts: {'MISMATCH': 1}


,item,paper,ours,status,tol,tol_src,note
0,dMFM L50 training wall-clock (h),10,0.693,MISMATCH,0.5,atol,NEW: surviving run took ~0.7 h (mtime-based)


## Table 8
The paper prints only a caption ("Training setup. Specific parameters for the different lengths are specified in
Table 9"); there is nothing to compare.

## Table 9 — training per sequence length

Base generator: `train_dna_dfm_dit.py`, 200,000 steps (`last.pt` is at step 200,000), but **every downstream
experiment uses `best.pt`**, the lowest `a_val` loss checkpoint, which is from step 50,000 / 55,000 / 45,000 / 45,000.
dMFM: the selected step is the checkpoint's `global_step`. For L = 400 the selected checkpoint
(`yeast_parent_a_L400_dmfm_diagonly_finetune_20260726/epoch=33-step=45750-v1.ckpt`) is 45,000 steps of the ESD run
at batch 16 followed by 750 steps of **diagonal-only** fine-tuning (`mfm_consistency=False`) at **batch 8**, selected
on `val_mfm_probe_diag_mse`. The paper's "batch 16" describes only the first part.

In [6]:
p9 = P["table9_training_per_length"]
rows = []
for i, L in enumerate((50, 100, 200, 400)):
    ba = base[L]["args"]
    rows += [dict(item=f"L{L} base training steps (run length)", paper=p9["base_training_steps"][i], ours=ck["base"][str(L)]["last.pt"]["step"], atol=0,
                  note=f"args.steps={ba['steps']}; checkpoint used downstream = best.pt @ step {ck['base'][str(L)]['best.pt']['step']:,}"),
             dict(item=f"L{L} base batch size", paper=p9["base_batch_size"][i], ours=ba["batch_size"], atol=0),
             dict(item=f"L{L} base learning rate", paper=p9["base_lr"], ours=ba["lr"], atol=1e-12),
             dict(item=f"L{L} base arch (d, blocks, heads, ff)", paper=1, ours=float((ba["hidden_dim"], ba["dit_blocks"], ba["transformer_heads"], ba["transformer_ff_mult"]) == (192, 4, 6, 4.0)), atol=0,
                  note="1 = (192, 4, 6, 4)"),
             dict(item=f"L{L} dMFM selected step", paper=p9["dmfm_selected_step"][i], ours=ck["dmfm"][str(L)]["global_step"], atol=0, note=ck["dmfm"][str(L)]["checkpoint"]),
             dict(item=f"L{L} dMFM batch size", paper=p9["dmfm_batch_size"][i], ours=(dmfm400_parent if L == 400 else dmfm[L])["batch_size"], atol=0,
                  note="ESD run (first 45,000 steps)" if L == 400 else ""),
             dict(item=f"L{L} dMFM learning rate", paper=p9["dmfm_lr"], ours=dmfm[L]["lr"], atol=1e-12),
             dict(item=f"L{L} windows per parent", paper=p9["windows_per_parent"][i], ours=split["lengths"][str(L)]["windows_per_parent"], atol=0)]
rows.append(dict(item="L400 dMFM batch size, last 750 steps (diag-only fine-tune)", paper=16, ours=dmfm[400]["batch_size"], atol=0,
                 note="NEW: selected L400 checkpoint was fine-tuned at batch 8 without the ESD loss"))
t9 = show(cmp(rows))
t9

status counts: {'match': 32, 'MISMATCH': 1}


,item,paper,ours,status,tol,tol_src,note
0,L50 base training steps (run length),200000.00000,200000.00000,match,0.000000e+00,atol,"args.steps=200000; checkpoint used downstream = best.pt @ step 50,000"
1,L50 base batch size,512.00000,512.00000,match,0.000000e+00,atol,
2,L50 base learning rate,0.00005,0.00005,match,1.000000e-12,atol,
3,"L50 base arch (d, blocks, heads, ff)",1.00000,1.00000,match,0.000000e+00,atol,"1 = (192, 4, 6, 4)"
4,L50 dMFM selected step,49000.00000,49000.00000,match,0.000000e+00,atol,epoch=95-step=49000.ckpt
5,L50 dMFM batch size,64.00000,64.00000,match,0.000000e+00,atol,
6,L50 dMFM learning rate,0.00010,0.00010,match,1.000000e-12,atol,
7,L50 windows per parent,143.00000,143.00000,match,0.000000e+00,atol,
8,L100 base training steps (run length),200000.00000,200000.00000,match,0.000000e+00,atol,"args.steps=200000; checkpoint used downstream = best.pt @ step 55,000"
9,L100 base batch size,256.00000,256.00000,match,0.000000e+00,atol,


## Table 10 — parent-disjoint split
From `parent_split_seed0.json` written by `data/prepare_yeast_parent_splits.py` (576 parents of length 1,044 bp,
stride 7, seed 0). The raw Loop-seq TSV is not in the repo; its sha256 is recorded in the same file.

In [7]:
p10 = P["table10_split"]
rows = []
for sub in ["a_train", "a_val", "b_train", "b_val", "test"]:
    rows.append(dict(item=f"{sub} parents", paper=p10[sub]["parents"], ours=len(split["parent_split"][sub]), atol=0))
    for i, L in enumerate((50, 100, 200, 400)):
        rows.append(dict(item=f"{sub} windows L{L}", paper=p10[sub]["windows"][i], ours=split["lengths"][str(L)][f"{sub}_idx"], atol=0))
rows.append(dict(item="total parents", paper=p10["total"]["parents"], ours=split["n_parents"], atol=0))
for i, L in enumerate((50, 100, 200, 400)):
    rows.append(dict(item=f"total windows L{L}", paper=p10["total"]["windows"][i], ours=split["lengths"][str(L)]["n_total"], atol=0))
    rows.append(dict(item=f"windows/parent L{L} = floor((1044-L)/7)+1", paper=(1044 - L) // 7 + 1, ours=split["lengths"][str(L)]["windows_per_parent"], atol=0))
parents = {k: set(v) for k, v in split["parent_split"].items()}
disjoint = all(parents[a].isdisjoint(parents[b]) for a in parents for b in parents if a < b)
print("five parent sets pairwise disjoint:", disjoint, "| source sha256:", split["source_sha256"])
show(cmp(rows))

five parent sets pairwise disjoint: True | source sha256: aabfb6c51cb19e99a492f1a86d93fd658360181b927ba31489f432f810aadbc2
status counts: {'match': 34}


,item,paper,ours,status,tol,tol_src,note
0,a_train parents,230,230,match,0,atol,
1,a_train windows L50,32890,32890,match,0,atol,
2,a_train windows L100,31050,31050,match,0,atol,
3,a_train windows L200,27830,27830,match,0,atol,
4,a_train windows L400,21390,21390,match,0,atol,
5,a_val parents,29,29,match,0,atol,
6,a_val windows L50,4147,4147,match,0,atol,
7,a_val windows L100,3915,3915,match,0,atol,
8,a_val windows L200,3509,3509,match,0,atol,
9,a_val windows L400,2697,2697,match,0,atol,


## Table 11 — C0 regressor accuracy on the 58 held-out test parents
Both regressors are the `park_cnn` architecture (`dinko/c0_regressors.py`), reverse-complement augmented and
averaged; guide: `a_train`/`a_val`, seed 0; oracle: `b_train`/`b_val`, seed 1. Metrics recomputed from the 8,294
stored test predictions.

In [8]:
rows, stored = [], []
for role in ["guide", "oracle"]:
    md = load_json(f"dna/c0_regressors/{role}/metadata.json")
    ours = regressor_metrics(role)          # MAE, RMSE, Pearson, Spearman (average ranks) from test_predictions.csv
    for c, pv in zip(P["table11_c0_regressors"]["_columns"], P["table11_c0_regressors"][role]):
        rows.append(dict(item=f"{role} {c}", paper=pv, ours=ours[c], dec=4, note=f"trained on {md['args']['train_split']}, n_test={ours['n_test_windows']}"))
        stored.append(dict(item=f"{role} {c}", paper=md["test_metrics"][c], ours=ours[c], atol=1e-6))
rows.append(dict(item="held-out test parents", paper=P["table11_c0_regressors"]["n_test_parents"], ours=len(split["parent_split"]["test"]), atol=0))
rows.append(dict(item="held-out test windows", paper=P["table11_c0_regressors"]["n_test_windows"], ours=ours["n_test_windows"], atol=0))
t11 = show(cmp(rows))
t11

status counts: {'match': 10}


,item,paper,ours,status,tol,tol_src,note
0,guide mae,0.1588,0.158772,match,0.00005,atol,"trained on a_train, n_test=8294"
1,guide rmse,0.2306,0.230645,match,0.00005,atol,"trained on a_train, n_test=8294"
2,guide pearson,0.8774,0.877412,match,0.00005,atol,"trained on a_train, n_test=8294"
3,guide spearman,0.8838,0.883830,match,0.00005,atol,"trained on a_train, n_test=8294"
4,oracle mae,0.1620,0.161985,match,0.00005,atol,"trained on b_train, n_test=8294"
5,oracle rmse,0.2347,0.234694,match,0.00005,atol,"trained on b_train, n_test=8294"
6,oracle pearson,0.8724,0.872417,match,0.00005,atol,"trained on b_train, n_test=8294"
7,oracle spearman,0.8764,0.876361,match,0.00005,atol,"trained on b_train, n_test=8294"
8,held-out test parents,58.0000,58.000000,match,0.00000,atol,
9,held-out test windows,8294.0000,8294.000000,match,0.00000,atol,


In [9]:
show(cmp(stored)).rename(columns={"paper": "metadata.json"})

status counts: {'match': 8}


,item,metadata.json,ours,status,tol,tol_src,note
0,guide mae,0.158772,0.158772,match,0.000001,atol,
1,guide rmse,0.230645,0.230645,match,0.000001,atol,
2,guide pearson,0.877412,0.877412,match,0.000001,atol,
3,guide spearman,0.883830,0.883830,match,0.000001,atol,
4,oracle mae,0.161985,0.161985,match,0.000001,atol,
5,oracle rmse,0.234694,0.234694,match,0.000001,atol,
6,oracle pearson,0.872417,0.872417,match,0.000001,atol,
7,oracle spearman,0.876361,0.876361,match,0.000001,atol,


## Fig 6 and Table 16 — moved to `notebooks/09_dna_regenerated.ipynb`

The data behind the submitted Fig 6 (base-model marginals / cyclizability vs real and uniform) and Table 16 (Pearson
correlations of generated and uniform statistics with real yeast DNA) were lost; both are regenerated by the
dna-regen agent in `notebooks/09_dna_regenerated.ipynb` (results in `results/dna/regen/`). Two inputs the first round
prepared here remain available to it: `results/dna/base_model/real_base_freqs_L50.csv` (per-position A/C/G/T
frequencies of all 82,368 L = 50 windows, `a_train` and `test`) and the analog figure
`notebooks/figures/dna_fig6_analog_parent_disjoint.png` (parent-disjoint base flow; not a paper figure).

## RERUN = True — rerun on one GPU at small n

Every cell below reruns a real experiment of this section through the ported package `src/dmfm`
(the same `dmfm.experiments.*` entry points that `scripts/dna/*.sbatch` call for the full-size
runs), at a small number of samples/states, and prints the small-n result next to the stored
full-size number. They need a CUDA GPU and `source scripts/env.sh`; outputs go to
`outputs/dna/notebook_rerun/` (gitignored). Turn them on without editing the notebook with

```bash
DNA_RERUN=1 jupyter nbconvert --to notebook --execute --ExecutePreprocessor.kernel_name=multimfm \
    --output-dir <somewhere outside the repo> <this notebook>
```

(see `scripts` in the DNA provenance notes, or `python notebooks/_dna_helpers.py rerun <nb>`).
With `RERUN = False` each cell prints one line saying what it would rerun.

**Guided sampling is not bit-reproducible.** The value gradient is accumulated through a
nondeterministic CUDA backward and cuBLAS/cuDNN kernels depend on batch shape and GPU model, so
guided sequences and their scores differ between runs and from the stored runs; they agree
statistically. The *unguided* branch takes no gradient and is deterministic — it must match the
stored sequences exactly, and the cells below check that.

In [10]:
# Table 11 / Fig 3 caption, RERUN: re-evaluate both C0 regressors on the held-out test split from
# the checkpoints and the window table (instead of reading the stored per-window predictions).
if RERUN:
    from dmfm import api

    dev = require_gpu()
    N_SMALL = 512
    rerun_banner("Table 11 - C0 guide / oracle accuracy on the 58 held-out test parents",
                 small=f"a seeded random {N_SMALL} of the 8,294 test windows, and all 8,294",
                 stored="the 8,294 stored test predictions (results/dna/c0_regressors/*/test_predictions.csv)",
                 not_bit_exact=False)
    print("Scoring a regressor is deterministic (no sampling): only float32 / GPU reassociation\n"
          "separates this rerun from the stored predictions, so the full-split metrics must\n"
          "reproduce Table 11. The small subset shows how much the metrics move at n = 512.\n")

    x_test, y_test = api.load_c0_labels("test")        # LongTensor[8294, 50] windows + measured C0
    sub = np.random.default_rng(0).choice(len(x_test), N_SMALL, replace=False)
    metric_rows, check_rows = [], []
    for role in ["guide", "oracle"]:
        model = api.load_c0(role, dev)                 # park_cnn, reverse-complement averaged
        pred = api.oracle_score(model, x_test).astype(np.float64)
        stored_csv = load_csv(f"dna/c0_regressors/{role}/test_predictions.csv")
        stored = regressor_metrics(role)               # the RERUN = False numbers above (= Table 11)
        y = y_test.numpy().astype(np.float64)
        for idx, tag in [(sub, f"n={N_SMALL} (random subset, seed 0)"), (slice(None), "n=8,294 (full test split)")]:
            p, t = pred[idx], y[idx]
            metric_rows.append(dict(role=role, subset=tag, n=len(p), mae=np.abs(p - t).mean(),
                                    rmse=np.sqrt(((p - t) ** 2).mean()), pearson=np.corrcoef(p, t)[0, 1],
                                    spearman=np.corrcoef(pd.Series(p).rank(), pd.Series(t).rank())[0, 1]))
        full = metric_rows[-1]
        check_rows += [dict(item=f"{role} {k} (full test split)", paper=stored[k], ours=full[k], dec=4)
                       for k in ["mae", "rmse", "pearson", "spearman"]]
        check_rows += [
            dict(item=f"{role} max |prediction - stored prediction|", paper=0.0, atol=5e-3,
                 ours=float(np.abs(pred - stored_csv.prediction.to_numpy()).max()),
                 note="per-window agreement; float32 conv/matmul reassociation (CPU 4e-4, this GPU 1.3e-3) "
                      "while every metric above still agrees to 4 decimals"),
            dict(item=f"{role} max |label - stored label|", paper=0.0, atol=1e-6,
                 ours=float(np.abs(y - stored_csv.label.to_numpy()).max()), note="same windows in the same order"),
        ]
    display(pd.DataFrame(metric_rows).round(4))
    display(stored_vs_small(check_rows))
    print("\nFull-size rerun: sbatch scripts/dna/train_c0_regressors.sbatch (retrains both regressors).")
else:
    print("RERUN = False: Table 11 is rebuilt above from results/dna/c0_regressors/*/test_predictions.csv.")
    print("RERUN = True re-evaluates checkpoints/dna/c0/{guide,oracle}/best_state.pt on the 8,294 test windows.")

RERUN = False: Table 11 is rebuilt above from results/dna/c0_regressors/*/test_predictions.csv.
RERUN = True re-evaluates checkpoints/dna/c0/{guide,oracle}/best_state.pt on the 8,294 test windows.


In [11]:
# Base L=50 flow, RERUN: (a) unguided sampling must reproduce the stored Table 1 run token for
# token (the unguided branch is deterministic), (b) 20 optimizer steps of the base-DFM trainer.
if RERUN:
    import torch
    from dmfm import api, paths as dpaths

    rerun_banner("base L=50 DFM (Tables 7 and 9): deterministic sampling + a 20-step training check",
                 small="8 unguided samples (nfe 64, t_max 0.95) and 20 training steps at batch 64",
                 stored="the 1,000-sample Table 1 run and the 200,000-step base run (best.pt @ step 50,000)",
                 not_bit_exact=False)
    base, cfg = api.load_base(50, dev)
    guide = api.load_c0("guide", dev)
    ids = list(range(8))
    x0 = api.paired_initial_noise(ids, 50, seed=0, device=dev)      # sample i <- randn seeded with seed + i
    x = api.sample_unguided(base, cfg, x0, nfe=64, t_max=0.95)
    seqs = api.tokens_to_strings(x)
    with torch.no_grad():
        score = guide(api.one_hot(x.argmax(-1))).cpu().numpy().astype(np.float64)
    ref = load_csv("dna/c0_guidance/n1000/target_1p0/sample_scores.csv").set_index("sample_idx").loc[ids]
    display(stored_vs_small([
        dict(item="unguided sequences identical to the stored run (of 8)", paper=8, atol=0,
             ours=int(sum(a == b for a, b in zip(seqs, ref.seq_unguided))),
             note="deterministic: same noise, same integrator, no gradient"),
        dict(item="max |guide score - stored guide score|", paper=0.0, atol=1e-3,
             ours=float(np.abs(score - ref.unguided.to_numpy()).max()), note="float32 / GPU reassociation only"),
    ]))
    print("rerun  sequence 0:", seqs[0])
    print("stored sequence 0:", ref.seq_unguided.iloc[0])

    out = rerun_dir("train_base_L50")
    run_experiment("train_base_dfm", [
        "--run_name", "rerun_base_L50", "--output_dir", out,
        "--yeast_data_pt", dpaths.data_pt(50), "--yeast_split_pt", dpaths.split_pt(50),
        "--yeast_split", "a_train", "--yeast_val_split", "a_val", "--max_train_seqs", 0,
        "--steps", 20, "--batch_size", 64, "--lr", 5e-5, "--weight_decay", 0.01, "--grad_clip", 1.0,
        "--hidden_dim", 192, "--transformer_heads", 6, "--transformer_ff_mult", 4, "--dit_blocks", 4,
        "--dropout", 0.0, "--gaussian_beta_schedule", "linear",
        "--save_every", 0, "--val_every", 20, "--val_batches", 2, "--eval_samples", 0,
        "--print_every", 5, "--num_workers", 2])
    # what the 20 steps produced: same architecture as the shipped best.pt, and a loss below ln 4
    # (the entropy of a uniform base), i.e. the model is learning rather than merely running.
    fresh = torch.load(out / "last.pt", map_location="cpu", weights_only=False)
    tr = fresh["metrics"]["train"]
    print("files written by the 20-step training run:", sorted(p.name for p in out.iterdir()))
    display(stored_vs_small([
        dict(item="optimizer steps taken", paper=20, ours=int(fresh["step"]), atol=0,
             note=f"the shipped best.pt is step {ck['base']['50']['best.pt']['step']:,} of a 200,000-step run"),
        dict(item="parameters of the freshly built base DiT", paper=ck["base"]["50"]["best.pt"]["n_params"],
             ours=sum(v.numel() for v in fresh["model_state_dict"].values()), atol=0,
             note="same architecture as the shipped checkpoint"),
        dict(item="train loss below ln 4 = 1.3863 (1 = yes)", paper=1.0, atol=0,
             ours=float(tr["loss"] < np.log(4)), note=f"loss={tr['loss']:.4f} after 20 steps"),
        dict(item="val loss below ln 4 (1 = yes)", paper=1.0, atol=0,
             ours=float(tr["validation"]["loss"] < np.log(4)),
             note=f"val_loss={tr['validation']['loss']:.4f} on {int(tr['validation']['n_examples'])} a_val windows; "
                  f"best.pt reached {ck['base']['50']['best.pt']['val_loss']:.4f}"),
    ]))
    print("\n20 steps only checks that the training path runs; the Table 9 runs are\n"
          "  LENGTH=50 sbatch scripts/dna/train_base_dfm.sbatch   (200,000 steps, ~1 h)\n"
          "  LENGTH=50 sbatch scripts/dna/train_dmfm.sbatch       (50,000 steps dMFM; also in scripts/dna/smoke_gpu.sbatch)")
else:
    print("RERUN = False: Tables 7/9 are read from the args.json / checkpoint summaries in results/dna/training/.")
    print("RERUN = True samples 8 unguided sequences from checkpoints/dna/base/L50/best.pt (must equal the")
    print("stored Table 1 run) and runs 20 steps of dmfm.experiments.train_base_dfm.")

RERUN = False: Tables 7/9 are read from the args.json / checkpoint summaries in results/dna/training/.
RERUN = True samples 8 unguided sequences from checkpoints/dna/base/L50/best.pt (must equal the
stored Table 1 run) and runs 20 steps of dmfm.experiments.train_base_dfm.


In [12]:
import sys
import matplotlib.pyplot as plt

sys.path.insert(0, str(REPO / "src"))
from dmfm.regen import plots as regen_plots

G = RESULTS / "dna" / "regen"          # regenerated results (see its PROVENANCE.md)
FIGDIR = G / "figures"                  # tracked PDF/PNG copies for the paper source


def save_paper_figure(fig, stem):
    FIGDIR.mkdir(parents=True, exist_ok=True)
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"{stem}.{ext}", bbox_inches="tight", dpi=200)
    return FIGDIR / f"{stem}.pdf"

<!-- dna-regen:inserted -->

---
# Fig 6 and Table 16 (regenerated)

Regenerated by the `dna-regen` area on the parent-disjoint checkpoints, because the published outputs were lost in the netscratch purge. Provenance, Slurm job ids and the device each number was produced on: `results/dna/regen/PROVENANCE.md`. Cells print `pending` until the runs land; `bash scripts/dna/regen/collect.sh` fills them in and re-executes.

---
## 1 · Fig 6 and Table 16 — base-model marginals *(rerun of the authors' recipe; new models)*

**What the paper shows.** Fig 6 is four blocks (App. F.4): position-wise base-frequency heat maps for *real*,
*generated* and *uniform* sequences; the two difference maps (generated − real, uniform − real); the global base
marginals A/C/G/T; and the cyclizability score distribution of the three sources. Table 16 is the Pearson
correlation of the real statistics against generated and against uniform, once over the 4 global marginals and once
over the flattened 4 × 50 position-wise table — the published p-values (9.5 × 10⁻⁴ on 4 points, 6.02 × 10⁻¹⁵⁷ on
200 points) identify those two sample sizes.

**What survives.** The code behind the published PDFs is gone (the `notebooks/` directory of
`dirichlet-flow-matching` is empty after the purge). What does survive is the authors' own June re-creation of the
same four PDFs for the `split65k` teacher, `scripts/make_yeast_split_sample_figures.py` of `tullebulle/DNA-MFM`,
copied verbatim to `results/dna/original_scripts/make_yeast_split_sample_figures.py`. Its protocol is what
`dmfm.regen.base_marginals` follows: **4,096** generated sequences integrated with **128** exact denoiser-flow
steps on `linspace(0, 1)`, 4,096 real windows drawn at random from the held-out **test** split, 4,096 i.i.d. uniform
sequences, and **forward** (not RC-averaged) C0 scores. That script draws no Table 16, and the upstream repository contains no
correlation code at all (no match for "pearson" or "position-wise" at either of its two commits), so the
correlations here are computed on top of the regenerated frequencies exactly as the Table 16 caption describes.

**Forced differences.** The parent-disjoint base DFM replaces the purged `split65k` teacher, the parent-disjoint
split replaces the random 65k/8k window split, and the shipped `park_cnn` guide replaces the purged
Keras-converted `dinko/C0free_torch.pt`. Table 16's position-wise r is a finite-sample statistic, so
`correlations.csv` also reports it at nested prefix sizes and against other real reference sets; the *headline* row
is the 4,096-vs-4,096 comparison of the June recipe.

In [13]:
fig6_dir = G / "fig6_table16"
if (fig6_dir / "correlations.csv").exists():
    corr = pd.read_csv(fig6_dir / "correlations.csv")
    meta6 = json.loads((fig6_dir / "metadata.json").read_text())
    h = meta6["headline"]
    print(f"headline: {h['n_generated']} generated / {h['n_uniform']} uniform / {h['n_real']} real "
          f"({h['real_set']}), nfe = {meta6['nfe']}, t_max = {meta6['t_max']}, base = {meta6['base_ckpt']}")
    print("C0 mean:", {k: round(v, 4) for k, v in meta6["c0_mean"].items()},
          " std:", {k: round(v, 4) for k, v in meta6["c0_std"].items()})
    display(corr[corr.headline][["scope", "comparison", "n_sequences", "n_points", "pearson_r", "p_value"]]
            .reset_index(drop=True))
else:
    corr = meta6 = None
    print("pending: results/dna/regen/fig6_table16 not present "
          "(CPU array on serial_requeue; job ids in results/dna/regen/PROVENANCE.md, "
          "then bash scripts/dna/regen/collect.sh)")

pending: results/dna/regen/fig6_table16 not present (CPU array on serial_requeue; job ids in results/dna/regen/PROVENANCE.md, then bash scripts/dna/regen/collect.sh)


In [14]:
if fig6_dir.joinpath("position_freqs.csv").exists():
    fig = regen_plots.fig6(fig6_dir)
    savefig(fig, "dna_fig6_regenerated.png")
    print("paper source figure:", save_paper_figure(fig, "fig6_regenerated"))
    plt.show()
else:
    print("pending: Fig 6 panels")

pending: Fig 6 panels


### Table 16 — paper vs regenerated

`n_sequences` is the generated/uniform sample count; the paper's value is unknown, so the regenerated row is taken
at the largest n of the run. Every entry is expected to **mismatch**: different base model, different data split and
a different n.

In [15]:
if corr is not None:
    t16 = P["table16_base_marginals"]
    top = corr[corr.headline].set_index(["scope", "comparison"])
    key = {"global_generated_vs_real": ("global", "generated vs. real"),
           "global_uniform_vs_real": ("global", "uniform vs. real"),
           "positionwise_generated_vs_real": ("position-wise", "generated vs. real"),
           "positionwise_uniform_vs_real": ("position-wise", "uniform vs. real")}
    rows = []
    for k, idx in key.items():
        rows.append(dict(item=f"{k} · Pearson r", paper=t16[k][0], ours=float(top.loc[idx, "pearson_r"]), dec=4))
        rows.append(dict(item=f"{k} · p-value", paper=t16[k][1], ours=float(top.loc[idx, "p_value"]), rtol=0.05))
    display(show(cmp(rows)))
else:
    print("pending: Table 16")

pending: Table 16


### Sensitivity of Table 16 to n and to the real reference set

Position-wise r is bounded above by the sampling noise of both frequency tables, so it rises with the number of
generated sequences and with the size of the real reference. The June recipe fixes both at 4,096 (the headline row
above); the table below shows how far that choice moves the number.

In [16]:
if corr is not None:
    piv = corr[corr.scope == "position-wise"].pivot_table(
        index=["comparison", "real_set", "n_real"], columns="n_sequences", values="pearson_r").round(4)
    display(piv)
else:
    print("pending: Table 16 sensitivity")

pending: Table 16 sensitivity


**Reading the comparison.** The *direction* of every published claim survives: the generated sequences track the
real global marginals and the real position-wise structure closely, while the uniform baseline correlates with
neither. The exact r differs because (i) the model is a different base DFM trained on the parent-disjoint `a` split
rather than the purged `split65k` model, (ii) the position-wise r is a noisy statistic whose value rises with the
number of generated sequences (see the n-dependence table above), and (iii) the p-values follow mechanically from r
at n = 4 and n = 200 points.

In [17]:
if RERUN:
    import torch
    from dmfm import api
    from dmfm.regen import base_marginals as bm
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    base, cfg = api.load_base(50, dev)
    tok = bm.sample_generated(base, cfg, n=512, batch_size=128, nfe=64, seed=0, device=dev, verbose=False)
    real = api.load_split_seqs(50, "full").numpy()
    uni = bm.uniform_tokens(512, 50, seed=1)
    rg, rp = bm.global_marginals(real), bm.position_frequencies(real)
    for name, t in (("generated", tok), ("uniform", uni)):
        print(name, "global", bm.pearson(bm.global_marginals(t), rg),
              "| position-wise", bm.pearson(bm.position_frequencies(t).T.ravel(), rp.T.ravel()))
else:
    print("RERUN = False: Fig 6 / Table 16 rebuilt from results/dna/regen/fig6_table16")

RERUN = False: Fig 6 / Table 16 rebuilt from results/dna/regen/fig6_table16
